<a href="https://colab.research.google.com/github/SafaaMahbub/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    for t in range(tries):
      r = requests.get(BASE,
                 params,
                 timeout=10)

      if r.status_code == 200:
        return pd.DataFrame(r.json()['daily'])
      if r.status_code not in RETRY_STATUSES:
        r.raise_for_status()
      if (t <tries-1):
        delay = (2 ** t)
        print(f'attempt {t + 1}; retrying in {delay}s')
        time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts')
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

I added a loop that attempts the request data up to tries times. I set a 10 second timeout. If the request is successful, we return the data as a dataframe and if it is not, we raise an error for non-retryable codes. For status codes that are retryable, we give a delay before trying again the request.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
df = get_weather('2026-09-05','2026-10-03')
used_fallback = False
wx = df[df['time'].isin(['2026-09-05','2026-09-11'])]

print('full data:',df)
print(df.shape)
print('selected dates:',wx)

full data:           time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
1   2026-09-06                27.2                1.6
2   2026-09-07                27.7                0.0
3   2026-09-08                29.4                0.0
4   2026-09-09                30.9                0.0
5   2026-09-10                34.2                3.2
6   2026-09-11                28.0               10.4
7   2026-09-12                27.5               21.5
8   2026-09-13                31.3                5.8
9   2026-09-14                28.7                0.3
10  2026-09-15                25.8                0.0
11  2026-09-16                29.7                0.0
12  2026-09-17                33.7                0.6
13  2026-09-18                30.0                1.6
14  2026-09-19                27.1                2.3
15  2026-09-20                30.5                0.1
16  2026-09-21                24.7                1.0
17  2026-09-22   

I pulled the data from september 5 to october 3 and did not use the fallback data. After that, I looked at the data and decided to filter the dates september 5 and semptember 11.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
  wx = FALLBACK.copy()
  wx['source'] = 'fallback sample'
else:
  wx = wx.copy()
  wx['source'] = 'open-meteo'
# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
6,2026-09-11,28.0,10.4,open-meteo


I did not use the fallback data, so I added a new column source that was filled with open-meteo

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO

wx['time'] = pd.to_datetime(wx['time'])
wx = wx.rename(columns={
    'time':'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',

})
display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
6,2026-09-11,28.0,10.4,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wettest_day = wx.loc[wx['precip_mm'].idxmax()]
difference = wettest_day['precip_mm'] - wx['precip_mm'].iloc[0]
print('The wettest day is :',wettest_day['date'], ' with ' , wettest_day['precip_mm'], "mm precipitation. The difference with other days is", difference, 'mm')

The wettest day is : 2026-09-11 00:00:00  with  10.4 mm precipitation. The difference with other days is 7.9 mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime
print('Endpoint: ', BASE)
print('Coordinates: latitude=38.08, longitude = -78.51')
print('Date Range: ', wx['date'].iloc[0], ' to ', wx['date'].iloc[1])
print('Pulled: ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data source: ' 'open-meteo (real data)')


Endpoint:  https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude=38.08, longitude = -78.51
Date Range:  2026-09-05 00:00:00  to  2026-09-11 00:00:00
Pulled:  2026-10-12 03:13
Data source: open-meteo (real data)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If the function returns an empty dataframe instead of rising an error, it means that we are not correctly pulling the data and that any further step would result in misleading information. Merging the data structure with 200 days of sales could result in missing weather values or an empty data set depending on what type of join is made. Nobody might catch the problem because the code runs correctly on a syntax level but not semantically and it may still look normal. The charts may seem blank but still not alert anyone looking at it because the code runs without an exception. This is incorrect because missing weather data could lead to wrong conclusions and the relationship between rain and revenue will  improperly be displayed on the charts.


### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# TODO
boston = get_weather('2026-09-05','2026-10-03',42.361145,-71.057083)
orlando = get_weather('2026-09-05','2026-10-03',28.54, -81.38)

boston['city'] = 'boston'
orlando['city']='orlando'

boston = boston[boston['time'].isin(['2026-09-05','2026-09-11'])]
orlando = orlando[orlando['time'].isin(['2026-09-05','2026-09-11'])]

locations = pd.concat([boston,orlando])

locations = locations[['city','time','temperature_2m_max','precipitation_sum']]

locations['time'] = pd.to_datetime(locations['time'])
locations = locations.rename(columns={
    'time':'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',

})
print(locations)


      city       date  temp_max  precip_mm
0   boston 2026-09-05      21.6        3.3
6   boston 2026-09-11      25.8        0.0
0  orlando 2026-09-05      32.1        3.9
6  orlando 2026-09-11      30.6       13.0
